# MediaPulse — Phase 3: Validated Data Cleaning Pipeline

**Project:** MediaPulse — Media Performance & Audience Analytics  
**Target Role:** Wipro Analyst L1 (Business Analyst / Data Analyst - Media)  
**Objective:** Transform raw, noisy YouTube trending observations into an audited, analysis-ready dataset with 100% mathematical reconciliation.

---

## Core Data Engineering & Cleaning Hierarchy:
1. **Schema Validation:** Enforce presence of all 16 required columns.
2. **Exact Duplicate Removal:** Eliminate technical scraper double-fetches (`4,263` rows).
3. **Identifier Quarantine:** Isolate unrecoverable `#NAME?` and `#VALUE!` records (`449` rows) to preserve relational integrity.
4. **Deterministic Snapshot Selection:** Resolve same-day metric updates on `(video_id, trending_date)` (`375` rows) by selecting the highest-view snapshot.
5. **Missing Value Imputation:** Handle missing descriptions without fabricating text.
6. **Date & Velocity Parsing:** Parse `YY.DD.MM` to ISO 8601 `YYYY-MM-DD` and calculate `days_to_trend`.
7. **Taxonomy Enrichment:** Resolve Category 29 as 'Nonprofits & Activism'.
8. **KPI Calculation:** Compute standard Engagement Rate: `((likes + comments) / views) * 100`.

### 1. Environment Setup & Module Imports

We import our modular `src.cleaning` and `src.metrics` packages.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to sys.path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import project modules
from src import config, profiling, cleaning, metrics

# Display settings
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

print('MediaPulse cleaning modules loaded successfully.')

MediaPulse cleaning modules loaded successfully.


### 2. Loading Raw Data & Category Taxonomy

We load `INvideos.csv` and `IN_category_id.json`.

In [2]:
raw_df, category_map = profiling.load_raw_dataset()
print(f"Raw Dataset Loaded: {len(raw_df):,} rows, {len(raw_df.columns)} columns.")
print(f"Taxonomy Categories Loaded: {len(category_map)} mappings.")

Raw Dataset Loaded: 37,352 rows, 16 columns.
Taxonomy Categories Loaded: 32 mappings.


### 3. Step 1: Schema Validation

Before applying transformations, we verify that all required columns exist.

In [3]:
is_valid, missing, unexpected = cleaning.validate_schema(raw_df)
print(f"Schema Valid: {is_valid}")
print(f"Missing Columns: {missing}")
print(f"Unexpected Columns: {unexpected}")
assert is_valid, "Schema validation failed!"

Schema Valid: True
Missing Columns: []
Unexpected Columns: []


### 4. Step 2: Exact Duplicate Rows Removal

Exact duplicates represent identical records scraped multiple times across all 16 columns.

In [4]:
df_exact_deduped, exact_dups_count = cleaning.deduplicate_exact_rows(raw_df)
print(f"Exact Duplicate Rows Removed: {exact_dups_count:,} ({exact_dups_count/len(raw_df)*100:.2f}%)")
print(f"Remaining Rows: {len(df_exact_deduped):,}")

Exact Duplicate Rows Removed: 4,263 (11.41%)
Remaining Rows: 33,089


### 5. Step 3: Quarantining Invalid Identifiers (`#NAME?`, `#VALUE!`)

Why we quarantine: 449 records have video IDs corrupted by Microsoft Excel formula errors. The original 11-character hash is permanently lost. Quarantining isolates them from video-level primary key joins while preserving them for auditability.

In [5]:
valid_df, quarantined_df = cleaning.quarantine_invalid_records(df_exact_deduped)
print(f"Quarantined Records: {len(quarantined_df):,} records.")
print(f"Valid ID Records: {len(valid_df):,} records.")

# Breakdown of quarantined identifiers
print("\nQuarantined Identifier Frequency:")
print(quarantined_df['video_id'].value_counts())

Quarantined Records: 449 records.
Valid ID Records: 32,640 records.

Quarantined Identifier Frequency:
video_id
#NAME?     442
#VALUE!      7
Name: count, dtype: int64


### 6. Step 4: Deterministic Same-Day Snapshot Selection

A video scraped multiple times on the same calendar day may have updated view counts (e.g., 50K in the morning, 80K in the evening). We sort descending by `views`, `likes`, `comment_count` to deterministically retain the latest/peak snapshot for that day.

In [6]:
clean_valid_df, sameday_dups_removed = cleaning.deduplicate_sameday_snapshots(valid_df)
print(f"Same-Day Snapshot Duplicates Removed: {sameday_dups_removed:,}")
print(f"Clean Valid Observations: {len(clean_valid_df):,}")

Same-Day Snapshot Duplicates Removed: 375
Clean Valid Observations: 32,265


### 7. Step 5: Handling Missing Descriptions

We impute missing descriptions with an empty string `""` and add a boolean flag `has_description`.

In [7]:
clean_valid_df = cleaning.handle_missing_values(clean_valid_df)
quarantined_df = cleaning.handle_missing_values(quarantined_df)

print(f"Remaining Null Descriptions: {clean_valid_df['description'].isnull().sum()}")
print(f"Videos with Descriptions: {clean_valid_df['has_description'].sum():,} ({clean_valid_df['has_description'].mean()*100:.2f}%)")

Remaining Null Descriptions: 0


Videos with Descriptions: 31,741 (98.38%)


### 8. Step 6: Temporal Standardization & Viral Latency

We convert dates to standard formats and compute `days_to_trend` (days from upload to trending list).

In [8]:
clean_valid_df = cleaning.standardize_dates(clean_valid_df)
quarantined_df = cleaning.standardize_dates(quarantined_df)

print(f"Trending Date Range: {clean_valid_df['trending_date'].min()} to {clean_valid_df['trending_date'].max()}")
print(f"Publish Date Range:  {clean_valid_df['publish_date'].min()} to {clean_valid_df['publish_date'].max()}")
print(f"Negative Duration Anomalies: {clean_valid_df['is_negative_duration'].sum()}")
print("\nDays to Trend Statistics:")
print(clean_valid_df['days_to_trend'].describe())

Trending Date Range: 2017-11-14 to 2018-06-14


Publish Date Range:  2017-05-27 to 2018-06-13
Negative Duration Anomalies: 0

Days to Trend Statistics:
count   32265.0000
mean        2.1129
std         2.1741
min         0.0000
25%         1.0000
50%         2.0000
75%         3.0000
max       221.0000
Name: days_to_trend, dtype: float64


### 9. Step 7: Taxonomy Enrichment & Category 29 Resolution

We map `category_id` to human-readable titles using the verified YouTube taxonomy.

In [9]:
clean_valid_df = cleaning.enrich_categories(clean_valid_df, category_map=category_map)
quarantined_df = cleaning.enrich_categories(quarantined_df, category_map=category_map)

print("Top Categories in Cleaned Data:")
print(clean_valid_df['category_title'].value_counts().head(5))

Top Categories in Cleaned Data:
category_title
Entertainment      14419
News & Politics     4619
Music               3137
Comedy              2869
People & Blogs      2331
Name: count, dtype: int64


### 10. Step 8: Engagement Metrics Calculation

We calculate:
- **Engagement Rate:** `((likes + comments) / views) * 100`
- **Total Interaction Rate:** `((likes + dislikes + comments) / views) * 100`
- **Like Ratio (Sentiment):** `(likes / (likes + dislikes)) * 100`

In [10]:
clean_valid_df = metrics.add_engagement_metrics(clean_valid_df)

print("Engagement Metrics Summary (Clean Valid Dataset):")
print(clean_valid_df[['engagement_rate', 'total_interaction_rate', 'like_ratio']].describe())

Engagement Metrics Summary (Clean Valid Dataset):
       engagement_rate  total_interaction_rate  like_ratio
count       32265.0000              32265.0000  32265.0000
mean            2.4232                  2.5678     85.6720
std             3.3633                  3.4254     17.2973
min             0.0000                  0.0000      0.0000
25%             0.4777                  0.5694     82.7500
50%             1.0527                  1.1734     90.8975
75%             3.0801                  3.2697     95.7516
max            77.4282                 77.6485    100.0000


### 11. Complete Data Reconciliation Audit

We prove that every single raw observation is accounted for with zero double-counting.

In [11]:
reconciliation_summary = pd.DataFrame({
    "Component": [
        "Raw Input Rows",
        "Exact Duplicate Rows Removed",
        "Quarantined Records (#NAME? / #VALUE!)",
        "Same-Day Redundant Snapshots Removed",
        "Final Cleaned Valid Records"
    ],
    "Count": [
        len(raw_df),
        exact_dups_count,
        len(quarantined_df),
        sameday_dups_removed,
        len(clean_valid_df)
    ],
    "Percentage (%)": [
        100.0,
        round(exact_dups_count / len(raw_df) * 100, 2),
        round(len(quarantined_df) / len(raw_df) * 100, 2),
        round(sameday_dups_removed / len(raw_df) * 100, 2),
        round(len(clean_valid_df) / len(raw_df) * 100, 2)
    ]
})

# Verify reconciliation identity
reconciled_total = exact_dups_count + len(quarantined_df) + sameday_dups_removed + len(clean_valid_df)
print(reconciliation_summary.to_string(index=False))
print(f"\nReconciled Sum: {reconciled_total:,} == Raw Total: {len(raw_df):,} -> {reconciled_total == len(raw_df)}")
assert reconciled_total == len(raw_df), "Reconciliation discrepancy detected!"

                             Component  Count  Percentage (%)
                        Raw Input Rows  37352        100.0000
          Exact Duplicate Rows Removed   4263         11.4100
Quarantined Records (#NAME? / #VALUE!)    449          1.2000
  Same-Day Redundant Snapshots Removed    375          1.0000
           Final Cleaned Valid Records  32265         86.3800

Reconciled Sum: 37,352 == Raw Total: 37,352 -> True
